# Scientific Validity of Stratified PPI++ for Mean Estimation

This notebook provides empirical evidence that GLIDE's **Stratified Prediction-Powered Inference (Stratified PPI++)** implementation is statistically valid.

**Setup:** We estimate the mean of a binary outcome (e.g., the hallucination rate of an AI system) over a population that is naturally partitioned into **strata** (e.g., language, domain, or data source). Within each stratum we have:
- A small set of **true labels** (`y_true`), expensive but unbiased
- A large set of **proxy labels** (`y_proxy`), cheap but potentially biased

Stratified PPI++ fits an optimal rectification weight **per stratum**, then combines stratum-level estimates with population-proportional weights. This yields confidence intervals that are:
1. **Valid** : they cover the true mean at the specified rate (e.g. 90% confidence)
2. **Shorter** : as compared to those obtained with true labels only, especially when proxy quality is strong in at least one stratum

We test these two claims empirically across a range of proxy/true correlation levels.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from glide.confidence_intervals import CLTConfidenceInterval
from glide.estimators import ClassicalMeanEstimator, StratifiedClassicalMeanEstimator, StratifiedPPIMeanEstimator
from glide.simulators import generate_stratified_binary_dataset

## Experiment Parameters

We fix all parameters up front so every section of this notebook uses a consistent setup. We define:

- `CONFIDENCE_LEVEL` : the confidence level at which we will compute confidence intervals.

- `N_STRATA` : number of strata.

- `N_TRUE` : number of human annotated samples **per stratum**.

- `N_PROXY` : number of proxy only labeled samples **per stratum**.

- `STRATUM_TRUE_MEANS` : per-stratum true mean values.

- `STRATUM_PROXY_MEANS` : per-stratum (biased) proxy mean values.

- `TRUE_MEAN` : the population-weighted true mean, used as the ground-truth target for coverage evaluation.

- `N_SEEDS` : number of simulations in Monte Carlo experiments.

> **Note on correlation bounds:** Depending on the values of `STRATUM_TRUE_MEANS` and `STRATUM_PROXY_MEANS`, extreme correlation values (close to -1 or 1) may not be possible. The correlation sweep is kept within safe limits for all strata.

Finally, we define the baseline estimation methods for comparison:

- `True only` : uses true human labels only — the gold standard for validity
- `Proxy only` : uses proxy labels only — biased but cheap
- `Stratified PPI++` : Stratified Prediction-Powered Inference with per-stratum power tuning

In [ ]:
CONFIDENCE_LEVEL = 0.9
N_STRATA = 4
N_TRUE = np.array([100, 150, 150, 100])  # per stratum; total labeled = 500
N_PROXY = np.array([400, 600, 600, 400])  # per stratum; total proxy only = 2000
STRATUM_TRUE_MEANS = np.array([0.55, 0.4, 0.55, 0.5])
STRATUM_PROXY_MEANS = np.array([0.5, 0.45, 0.6, 0.55])
N_SEEDS = 1000

# Population weights: w_k = (N_TRUE_k + N_PROXY_k) / sum(N_TRUE + N_PROXY)
total_per_stratum = N_TRUE + N_PROXY
total = np.sum(total_per_stratum)
STRATUM_WEIGHTS = total_per_stratum / total

# Population-weighted true mean (ground truth for coverage validation)
TRUE_MEAN = np.sum(STRATUM_WEIGHTS * STRATUM_TRUE_MEANS)

METHODS = ["True only", "Proxy only", "Stratified PPI++"]

# Correlation sweep — kept within feasible range for both strata
correlations = np.arange(0.1, 0.95, 0.1)
n_correlations = len(correlations)
correlations_lmh = [
    correlations[n_correlations // 4],
    correlations[n_correlations // 2],
    correlations[3 * n_correlations // 4],
]  # low, medium and high values
corr_labels = ["Low", "Medium", "High"]

print(f"Population-weighted TRUE_MEAN = {TRUE_MEAN:.3f}")
print(f"Stratum weights: {STRATUM_WEIGHTS}")

## Data Generation

We use `generate_stratified_binary_dataset` to simulate a stratified evaluation scenario.
It generates correlated binary labels `y_true` and `y_proxy` for `N_TRUE[k]` samples per stratum, plus `N_PROXY[k]` samples per stratum with only `y_proxy` available. It returns **three arrays**: `y_true` (has NaNs for unlabeled samples), `y_proxy` (available for all samples), and `groups` (integer stratum identifiers).

The `correlation` parameter controls the Pearson correlation between the true and proxy labels on the labeled subset **within each stratum**. In the sweep below, both strata receive the same correlation value.

In [ ]:
# Single example dataset for illustration
y_true, y_proxy, groups = generate_stratified_binary_dataset(
    n=N_TRUE.tolist(),
    N=N_PROXY.tolist(),
    true_mean=STRATUM_TRUE_MEANS.tolist(),
    proxy_mean=STRATUM_PROXY_MEANS.tolist(),
    correlation=[0.8] * N_STRATA,
    random_seed=42,
)

n_labeled = int(np.sum(~np.isnan(y_true)))
n_unlabeled = len(y_true) - n_labeled

print(f"Total samples: {len(y_true)}")
print(f"Labeled samples: {n_labeled}")
print(f"Unlabeled samples: {n_unlabeled}")

In the following sections, we perform Monte Carlo experiments to estimate confidence interval width and coverage.

This consists in running `N_SEEDS` simulations where we generate stratified data, compute a confidence interval and measure its width each time. We end up with `N_SEEDS` sample values for each measured quantity.

## Inference Results

We compare three estimation methods:

| Estimation method | Data used | Notes |
|--------|-----------|-------|
| **True only** | `y_true` | Classical CLT Confidence Interval, the gold standard for validity |
| **Proxy only** | `y_proxy` | Biased, cheap but wrong |
| **Stratified PPI++** | `y_true` + `y_proxy` (rectified per stratum) | Best of both worlds, valid and efficient |

The function below computes means and standard deviations using all three estimation methods.

In [ ]:
def generate_estimates(y_true, y_proxy, groups):
    """Return mean and std for all three estimation methods."""
    # --- Stratified PPI++ ---
    stratified_ppi_result = StratifiedPPIMeanEstimator().estimate(
        y_true,
        y_proxy,
        groups,
        confidence_level=CONFIDENCE_LEVEL,
    )

    # --- Classical baselines ---
    estimator = StratifiedClassicalMeanEstimator()
    true_only_result = estimator.estimate(y_true, groups, confidence_level=CONFIDENCE_LEVEL)
    proxy_only_result = estimator.estimate(y_proxy, groups, confidence_level=CONFIDENCE_LEVEL)

    return {
        "True only": {
            "mean": true_only_result.mean,
            "std": true_only_result.std,
        },
        "Proxy only": {
            "mean": proxy_only_result.mean,
            "std": proxy_only_result.std,
        },
        "Stratified PPI++": {
            "mean": stratified_ppi_result.mean,
            "std": stratified_ppi_result.std,
            "ess": stratified_ppi_result.effective_sample_size,
        },
    }

`StratifiedPPIMeanEstimator` splits the samples by `stratum_id`, computes a power-tuned PPI++ estimate within each stratum, and combines them with population-proportional weights. `StratifiedClassicalMeanEstimator` implements conventional mean estimation using true labels only but partitions by stratum to compute the variance.

The three next functions implement the Monte Carlo verification:

- `monte_carlo_simulation` simulates stratified data for a single correlation level, applies each method and returns their outputs for each simulation.

- `compute_hits` computes how often each method's confidence interval contains `TRUE_MEAN`.

- `coverage_with_errbar` estimates the mean coverage and its confidence interval.

In [ ]:
def monte_carlo_simulation(correlation: float, n_seeds=N_SEEDS):
    """Single Monte Carlo loop: cache per-seed mean, std, and ESS for each estimation method."""
    means = {method: np.zeros(n_seeds) for method in METHODS}
    stds = {method: np.zeros(n_seeds) for method in METHODS}
    ess = np.zeros(n_seeds)

    for seed in range(n_seeds):
        y_true, y_proxy, groups = generate_stratified_binary_dataset(
            n=N_TRUE.tolist(),
            N=N_PROXY.tolist(),
            true_mean=STRATUM_TRUE_MEANS.tolist(),
            proxy_mean=STRATUM_PROXY_MEANS.tolist(),
            correlation=[correlation] * N_STRATA,
            random_seed=seed,
        )

        estimates = generate_estimates(y_true, y_proxy, groups)
        for method in METHODS:
            means[method][seed] = estimates[method]["mean"]
            stds[method][seed] = estimates[method]["std"]
        ess[seed] = estimates["Stratified PPI++"]["ess"]

    stats = {method: {"means": means[method], "stds": stds[method]} for method in METHODS}
    stats["Stratified PPI++"]["ess"] = ess
    return stats


def compute_hits(stats, confidence_level):
    """Return per-seed hit indicators {method: float array} at the given confidence level."""
    hits = {}
    for method in METHODS:
        ci = CLTConfidenceInterval(
            mean=stats[method]["means"], std=stats[method]["stds"], confidence_level=confidence_level
        )
        hits[method] = np.asarray((ci.lower_bound <= TRUE_MEAN) & (TRUE_MEAN <= ci.upper_bound), dtype=float)
    return hits


def coverage_with_errbar(hits, confidence_level):
    """Estimate empirical coverage + Confidence Interval via ClassicalMeanEstimator
    on per-seed hit indicators."""
    estimator = ClassicalMeanEstimator()
    r = estimator.estimate(hits, confidence_level=confidence_level)
    return r.mean, r.confidence_interval.lower_bound, r.confidence_interval.upper_bound

## Coverage Validity

A confidence interval is **valid** if it reliably captures the true value. For example, a 90% confidence interval is valid if, when you repeat the experiment many times and compute a new interval each time, approximately 90% of those intervals contain the true value.

We check this empirically via a Monte Carlo experiment and count how often each method's confidence interval covers `TRUE_MEAN`.

> **Key question:** Does Stratified PPI++ maintain valid coverage, or does it sacrifice it for shorter intervals?

### Coverage vs confidence level for three correlation levels

We sweep the confidence level from 0.55 to 0.95 and plot the observed coverage.
For a valid estimation method, the dots should fall on or above the black diagonal $y = \text{confidence level}$.

We do this for **low**, **medium** and **high** proxy correlation.

In [ ]:
# Run Monte Carlo simulations for each correlation level
raw_stats = {correlation: monte_carlo_simulation(correlation) for correlation in correlations}

confidence_levels = np.arange(0.55, 1.00, 0.05)

# Derive coverage for every (correlation, confidence_level) pair
coverages_confidence_intervals = {}
for correlation in correlations_lmh:
    coverages_confidence_intervals[correlation] = {}
    for confidence_level in confidence_levels:
        hits = compute_hits(raw_stats[correlation], confidence_level)
        coverages_confidence_intervals[correlation][confidence_level] = dict()
        for method in METHODS:
            coverage_confidence_interval = coverage_with_errbar(hits[method], confidence_level=confidence_level)
            coverages_confidence_intervals[correlation][confidence_level][method] = coverage_confidence_interval

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
colors = {"True only": "steelblue", "Stratified PPI++": "darkorange", "Proxy only": "red"}

for ax, correlation, label in zip(axes, correlations_lmh, corr_labels):
    ax.plot(confidence_levels, confidence_levels, color="black", lw=1.5, linestyle="--", label="Ideal")
    for method in METHODS:
        mean_ci = np.array([coverages_confidence_intervals[correlation][cl][method] for cl in confidence_levels])
        mean = mean_ci[:, 0]
        lo = mean_ci[:, 1]
        hi = mean_ci[:, 2]

        ax.plot(confidence_levels, mean, marker="o", color=colors[method], label=method)
        ax.fill_between(confidence_levels, lo, hi, alpha=0.15, color=colors[method])

    ax.set_xlabel("Target confidence level")
    ax.set_ylabel("Observed coverage")
    ax.set_title(f"{label} correlation (${round(correlation, 2)}$)")
    ax.legend(fontsize=8)
    ax.set_xlim(0.5, 1.0)
    ax.set_ylim(0.5, 1.0)

fig.suptitle("Empirical coverage vs target confidence level", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

Both **Stratified PPI++** and **True only** track the diagonal closely across all correlation levels, confirming that Stratified PPI++ achieves valid coverage regardless of proxy quality. The Proxy only method is far from the diagonal because it uses biased data so that its coverage is invalid.

---

### Coverage vs correlation for fixed confidence level

We now fix the confidence level at 90% and vary the proxy-true correlation from 0.1 to 0.8.
This shows that Stratified PPI++ validity does not degrade as the proxy becomes weaker.

In [ ]:
coverage_by_corr = {}  # {correlation: {method: observed mean coverage}}
coverage_ci_by_corr = {}  # {correlation: {method: (lower, upper) Confidence Interval on coverage}}

for correlation in correlations:
    hits = compute_hits(raw_stats[correlation], CONFIDENCE_LEVEL)
    coverage_by_corr[correlation] = {}
    coverage_ci_by_corr[correlation] = {}
    for method in METHODS:
        mean_cov, lo, hi = coverage_with_errbar(hits[method], CONFIDENCE_LEVEL)
        coverage_by_corr[correlation][method] = mean_cov
        coverage_ci_by_corr[correlation][method] = (lo, hi)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
method_colors = {"True only": "steelblue", "Proxy only": "forestgreen", "Stratified PPI++": "darkorange"}

for method in ["True only", "Stratified PPI++"]:
    obs = np.array([coverage_by_corr[correlation][method] for correlation in correlations])
    ci_bounds = np.array([coverage_ci_by_corr[correlation][method] for correlation in correlations])
    lo = ci_bounds[:, 0]
    hi = ci_bounds[:, 1]

    ax.plot(correlations, obs, marker="o", color=method_colors[method], label=method)
    ax.fill_between(correlations, lo, hi, alpha=0.15, color=method_colors[method])


ax.axhline(y=CONFIDENCE_LEVEL, color="red", linestyle="--", lw=2, label=f"Target coverage {CONFIDENCE_LEVEL:.0%}")
ax.set_xlabel("Proxy–true correlation")
ax.set_ylabel("Observed coverage")
ax.set_title(f"Coverage vs correlation  (confidence level = {CONFIDENCE_LEVEL:.0%})")
ax.set_xlim(0, 0.95)
ax.set_ylim(0.8, 1.0)
ax.legend()
plt.tight_layout()
plt.show()

Note that **Proxy only** is not plotted because the proxy is biased (proxy mean ≠ true mean in each stratum). Therefore it has invalid coverage (close to 0) whereas **Stratified PPI++** and **True only** remain valid across all correlation levels.

---

## Confidence Interval Width

Coverage validity is necessary but not sufficient — we also want **short** intervals. Stratified PPI++'s promise is that by leveraging the unlabeled proxy data with per-stratum rectification, it achieves the same validity as using labeled data alone, but with a shorter interval when the proxy is informative.

We report the **mean** and a **percentile band** to capture variability.

In [ ]:
width_by_corr = {}
for correlation in correlations:
    width_by_corr[correlation] = {}
    for method in METHODS:
        ci = CLTConfidenceInterval(
            mean=raw_stats[correlation][method]["means"],
            std=raw_stats[correlation][method]["stds"],
            confidence_level=CONFIDENCE_LEVEL,
        )
        width_by_corr[correlation][method] = ci.upper_bound - ci.lower_bound

The shaded bands show a confidence interval across Monte Carlo runs at the fixed `CONFIDENCE_LEVEL`.
At high correlation, Stratified PPI++'s confidence interval should be substantially narrower than true-only.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_methods_w = ["True only", "Stratified PPI++"]
colors_w = {"True only": "steelblue", "Stratified PPI++": "darkorange"}

# Compute percentiles based on CONFIDENCE_LEVEL
lower_percentile = round(((1 - CONFIDENCE_LEVEL) / 2) * 100)
upper_percentile = 100 - lower_percentile

for method in plot_methods_w:
    means_w = [np.mean(width_by_corr[correlation][method]) for correlation in correlations]
    q_lower = [np.percentile(width_by_corr[correlation][method], lower_percentile) for correlation in correlations]
    q_upper = [np.percentile(width_by_corr[correlation][method], upper_percentile) for correlation in correlations]
    ax.plot(correlations, means_w, marker="o", label=method, color=colors_w[method])
    ax.fill_between(correlations, q_lower, q_upper, alpha=0.15, color=colors_w[method])

ax.set_xlabel("Proxy–true correlation")
ax.set_ylabel("Confidence Interval width")
ax.set_title(
    f"Confidence interval width vs correlation  (confidence level = {CONFIDENCE_LEVEL:.0%})\n"
    f"Solid = mean, shaded = {lower_percentile:.0f}th–{upper_percentile:.0f}th percentile"
)
ax.set_xlim(0.05, 0.95)
ax.legend()
plt.tight_layout()
plt.show()

As expected, Stratified PPI++'s interval width decreases with increasing correlation. The per-stratum power tuning allows the estimator to leverage proxy data only where it is informative, further reducing interval width compared to a single global rectification.

---

## Effective Sample Size

A natural summary of Stratified PPI++'s efficiency gain is the **Effective Sample Size (ESS)**: it is the number of samples needed by the True only estimation method to achieve the same confidence interval width as Stratified PPI++ with the current sample size.

Since confidence interval width $\propto 1/\sqrt{n}$, we can estimate ESS empirically as:

$$\text{ESS} = N_{\text{true}} \times \left(\frac{\bar{w}_{\text{True only}}}{\bar{w}_{\text{Stratified PPI++}}}\right)^2,$$

where $\bar{w}_{\text{True only}}$ and $\bar{w}_{\text{Stratified PPI++}}$ are the confidence interval widths for True only and Stratified PPI++ respectively.

When the correlation is zero, ESS $\approx N_{\text{labeled}}$ (no gain)$.~$ As the correlation approaches $1,~$ ESS grows. Stratified PPI++ can be equivalent to having over **twice more** labeled samples.

In [ ]:
ess_mean = [np.mean(raw_stats[correlation]["Stratified PPI++"]["ess"]) for correlation in correlations]

ess_q_lower = [
    np.percentile(raw_stats[correlation]["Stratified PPI++"]["ess"], lower_percentile) for correlation in correlations
]
ess_q_upper = [
    np.percentile(raw_stats[correlation]["Stratified PPI++"]["ess"], upper_percentile) for correlation in correlations
]

In [ ]:
n_true_total = np.sum(N_TRUE)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(correlations, ess_mean, marker="o", color="darkorange", label="Stratified PPI++ ESS (mean)")
ax.fill_between(
    correlations,
    ess_q_lower,
    ess_q_upper,
    alpha=0.15,
    color="darkorange",
    label=f"{lower_percentile:.0f}th–{upper_percentile:.0f}th percentile",
)
ax.axhline(y=n_true_total, color="steelblue", linestyle="--", lw=2, label=f"Baseline (True only, n={n_true_total})")
ax.set_xlabel("Proxy–true correlation")
ax.set_ylabel("Effective sample size")
ax.set_title("Stratified PPI++ effective sample size vs proxy correlation")
ax.set_xlim(0.05, 0.95)
ax.legend()
plt.tight_layout()
plt.show()

## Summary

This notebook has empirically validated that GLIDE's Stratified PPI++ implementation satisfies two key statistical properties:

| Property | Result |
|----------|--------|
| **Coverage validity** | Stratified PPI++ achieves the nominal coverage across all correlation levels and confidence levels tested |
| **Efficiency** | Stratified PPI++ produces shorter confidence intervals than labeled-only whenever correlation is positive, with the gain growing with correlation |

Crucially, the biased baseline (**Proxy only**) fails the coverage test. It appears precise but is systematically wrong. Stratified PPI++ avoids this by correcting for proxy bias using the labeled subset **within each stratum**.

The ESS analysis shows that with moderate proxy correlation, Stratified PPI++ is equivalent to having significantly more labeled data — a substantial practical gain in scenarios where true annotation is expensive. By fitting a separate rectification weight per stratum, the estimator can fully exploit informative strata while gracefully degrading toward the labeled-only estimate in strata where the proxy is weak.